# Phase 6 - Notebook 02: Temporal Consistency & Dynamic 3DGS

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase6/02_temporal_consistency.ipynb)

---

## 学习目标

到这个笔记本结束，你将理解：
1. 从静态到动态场景的扩展
2. 逐帧高斯 vs 变形场
3. 4D Gaussian Splatting 的参数化
4. 时序正则化损失
5. 动态物体的检测与分离

**预计时间**：75 分钟

**先置条件**：Phase 1-5 + Phase 6-00, 01

---

## 1. 从静态到动态：问题描述

### 静态 3DGS 的假设

所有前面的笔记本都假设场景**不变**：
- Gaussian 参数（位置、尺度、旋转）固定
- 只有相机在移动
- 优化目标：使渲染与观察匹配

```
帧 0      帧 1      帧 2
┌──┐      ┌──┐      ┌──┐
│  │      │  │      │  │
│GS│(固定) │GS│(固定) │GS│(固定)
│  │      │  │      │  │
└──┘      └──┘      └──┘
 ↑         ↑         ↑
 相机位置  相机位置  相机位置
 变化      变化      变化
```

### 动态场景的挑战

```
帧 0       帧 1       帧 2
┌──┐      ┌──┐      ┌──┐
│  │      │  │      │  │
│GS│━━━━━►│GS'│━━━━━►│GS''│ Gaussians 变化！
│  │      │  │      │  │
└──┘      └──┘      └──┘
 ↑         ↑         ↑
 相机      相机      相机
 + 物体移动  + 物体移动  + 物体移动
```

**新增的自由度**：
- 每帧 Gaussian 数量可能不同
- Gaussian 参数随时间变化
- 新物体出现，旧物体消失
- 自遮挡（occlusion）变化

**关键问题**：
1. 如何表示时间上的变化？
2. 如何约束运动的平滑性？
3. 如何处理出现/消失的物体？
4. 如何在长序列中保持一致性？

In [ ]:
import sys
sys.path.insert(0, '../..')
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Ellipse

# 动态场景示意
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 绘制三帧，Gaussians 动起来
for frame_idx, ax in enumerate(axes):
    # 背景
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    
    # 相机位置
    cam_pos = frame_idx * 0.5
    ax.scatter(cam_pos, -2.5, s=200, marker='^', c='red', label='Camera')
    
    # Gaussians（移动的物体）
    np.random.seed(42)
    n_gaussians = 5
    
    for i in range(n_gaussians):
        # 物体位置随时间变化
        x = np.sin(frame_idx * 0.5 + i) * 1.5
        y = np.cos(frame_idx * 0.3 + i) * 1.5
        
        # Gaussian 参数
        scale = 0.3 + 0.1 * np.sin(frame_idx * 0.2 + i)
        rotation = frame_idx * 15 + i * 30
        
        # 绘制
        color = plt.cm.tab10(i)
        ellipse = Ellipse((x, y), scale*2, scale, angle=rotation,
                         facecolor=color, alpha=0.5, edgecolor='black', linewidth=2)
        ax.add_patch(ellipse)
    
    ax.set_title(f'Frame {frame_idx}', fontsize=12, fontweight='bold')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    
    if frame_idx == 0:
        ax.legend(loc='upper left')

fig.suptitle('Dynamic Scene: Gaussians Change Over Time', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n动态 3DGS 的关键差异：")
print("┌─────────────────────────────────────────────────────────┐")
print("│ 静态 3DGS           │ 动态 3DGS                           │")
print("├─────────────────────┼───────────────────────────────────┤")
print("│ 参数固定            │ 参数随时间变化                      │")
print("│ 优化：[μ, Σ, α, c] │ 优化：[μ(t), Σ(t), α(t), c(t)]     │")
print("│ 约束：无时序约束    │ 约束：平滑性、动态性                │")
print("│ 场景：室内、户外    │ 场景：视频、真实世界                │")
print("└─────────────────────┴───────────────────────────────────┘")

## 2. 表示动态的三种方式

### 方式 1: 逐帧高斯（Per-frame Gaussians）

**思想**：每一帧都维护一个独立的 Gaussian 集合

```
Frame 0: G_0 = {g_1^0, g_2^0, ..., g_N0^0}
Frame 1: G_1 = {g_1^1, g_2^1, ..., g_N1^1}
Frame 2: G_2 = {g_1^2, g_2^2, ..., g_N2^2}

其中 N_i 可能不同！
```

**优点**：
- 简单易实现
- 灵活处理物体出现/消失
- 每帧独立优化

**缺点**：
- 无时序约束 → 闪烁
- 大量重复的Gaussians
- 难以保证一致性
- 参数爆炸（N × T）

### 方式 2: 变形场（Deformation Field）

**思想**：维护基础 Gaussian 集合，用变形场表示运动

```
基础集合（t=0）: G_base = {g_1, g_2, ..., g_N}

变形场：D(x, t) → Δx

任意时刻的位置:
  μ(t) = μ_base + D(μ_base, t)
```

**优点**：
- 参数有限（只需学习 D）
- 自然的平滑约束
- 物理直观

**缺点**：
- 难以处理拓扑变化（物体出现/消失）
- 变形场的学习可能困难
- 大变形时效果差

### 方式 3: 4D 高斯（4D Gaussians）

**思想**：直接在 4D 空间（x, y, z, t）定义高斯

```
4D 高斯参数：
  - 均值：μ = [μ_x, μ_y, μ_z, μ_t] ∈ ℝ^4
  - 协方差：Σ ∈ ℝ^4×4（对称正定）
  - 其他：α, c（不变）

投影到 3D（给定时刻 t）：
  取 μ 和 Σ 的时间分量
  投影到 3D 空间和高斯参数
```

**优点**：
- 统一框架：时空对称
- 自然的连续性
- 一致的光栅化（沿时间整合）

**缺点**：
- 参数增多（4×4 协方差矩阵）
- 光栅化复杂（4D 投影到 2D）
- 数值不稳定性

In [ ]:
# 三种表示方式对比
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 方式 1: 逐帧高斯
ax = axes[0]
t = np.linspace(0, 1, 3)
for i, ti in enumerate(t):
    y = i
    # 每帧不同的高斯数量
    n_gauss = 4 + i
    for j in range(n_gauss):
        ax.scatter(j, y, s=100, c='steelblue', alpha=0.6)
    ax.plot([-0.5, n_gauss - 0.5], [y, y], 'k-', alpha=0.3)

ax.set_title('方式 1: Per-frame Gaussians', fontsize=11, fontweight='bold')
ax.set_xlabel('Gaussian Index')
ax.set_ylabel('Time Frame')
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['t=0', 't=1', 't=2'])
ax.text(3, -0.5, '问题：无约束、参数多', fontsize=9, style='italic', color='red')

# 方式 2: 变形场
ax = axes[1]
# 基础集合
for i in range(4):
    ax.scatter(1.5, i, s=150, c='green', marker='o', alpha=0.6, label='Base' if i == 0 else '')

# 变形后的位置
t_samples = [0, 1, 2]
for frame_idx, t_val in enumerate(t_samples):
    offset = 0.3 * (frame_idx - 1)
    for i in range(4):
        x = 1.5 + offset
        y = i + 0.1 * np.sin(t_val + i)
        ax.scatter(x, y, s=80, c='orange', alpha=0.5, marker='x')
    if frame_idx > 0:
        ax.arrow(1.5, -0.5, offset, 0, head_width=0.1, head_length=0.05, fc='gray', ec='gray')

ax.set_title('方式 2: Deformation Field', fontsize=11, fontweight='bold')
ax.set_xlabel('Deformed Position')
ax.set_ylabel('Gaussian Index')
ax.set_ylim(-1, 4.5)
ax.text(0.5, -0.8, '优势：参数少、平滑', fontsize=9, style='italic', color='green')

# 方式 3: 4D 高斯
ax = axes[2]
# 显示 3D 空间中的轨迹
t = np.linspace(0, 1, 10)
x = np.sin(2*np.pi*t)
y = np.cos(2*np.pi*t)

ax.plot(x, y, 'b-', linewidth=2, label='4D Gaussian trajectory')
for ti, xi, yi in zip(t[::2], x[::2], y[::2]):
    circle = plt.Circle((xi, yi), 0.1, fill=False, edgecolor='purple', linewidth=1.5, linestyle='--')
    ax.add_patch(circle)
    ax.scatter(xi, yi, s=50, c='purple', alpha=0.6)

ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.set_title('方式 3: 4D Gaussians', fontsize=11, fontweight='bold')
ax.set_xlabel('X (space)')
ax.set_ylabel('Y (space)')
ax.legend()
ax.text(-1.3, -1.3, '优势：统一框架', fontsize=9, style='italic', color='blue')

plt.tight_layout()
plt.show()

print("\n三种表示方式对比：")
print("┌─────────────────┬──────────┬────────────┬──────────┐")
print("│ 特性            │ Per-frame│ Deformation│ 4D       │")
print("├─────────────────┼──────────┼────────────┼──────────┤")
print("│ 参数数量        │ N×T      │ N          │ N        │")
print("│ 时间一致性      │ 差       │ 好         │ 好       │")
print("│ 拓扑变化        │ 容易     │ 困难       │ 困难     │")
print("│ 光栅化复杂度    │ 低       │ 中等       │ 高       │")
print("│ 实现难度        │ 简单     │ 中等       │ 困难     │")
print("└─────────────────┴──────────┴────────────┴──────────┘")

## 3. 时序正则化约束

无论采用哪种表示，关键是**约束运动的平滑性和物理合理性**。

### 约束类型

#### 1. 平滑约束（Smoothness）

```
鼓励 Gaussian 运动平稳，不要突跳跃：

L_smooth = Σ_i ||μ(t+1) - μ(t)|| ²  （一阶差分）
                 或
         Σ_i ||(μ(t+1) - μ(t)) - (μ(t) - μ(t-1))||²  （二阶差分）
```

#### 2. 刚体运动约束（Rigidity）

```
对于静态物体部分，鼓励同一物体的 Gaussians 保持相对位置：

L_rigid = Σ_{i,j∈物体} ||d_ij(t) - d_ij(0)||²

其中 d_ij = ||μ_i - μ_j|| 是两个 Gaussian 之间的距离
```

#### 3. 能量守恒（Physics）

```
虽然不强制物理模拟，但可以鼓励合理的加速度：

L_accel = Σ_i ||(μ(t+1) - μ(t)) - (μ(t) - μ(t-1))||²
```

#### 4. 光度一致性（Photometric）

```
RGB 颜色在物体表面应该缓慢变化：

L_color = Σ_i ||c(t+1) - c(t)||²  （RGB 平滑）
```

### 总损失函数

```
L_total = L_photo  （主要：渲染与观察匹配）
        + λ_smooth × L_smooth  （时间平滑）
        + λ_rigid × L_rigid    （刚体运动）
        + λ_accel × L_accel    （加速度）
        + λ_color × L_color    （颜色一致）
```

In [ ]:
# 时序正则化的效果演示
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 模拟一个 Gaussian 的轨迹
t = np.linspace(0, 2*np.pi, 20)

# 真实平滑轨迹
x_true = np.sin(t)
y_true = np.cos(t)

# 有噪声的观察
np.random.seed(42)
x_noisy = x_true + np.random.randn(len(t)) * 0.15
y_noisy = y_true + np.random.randn(len(t)) * 0.15

# 无约束优化（逐帧独立）
ax = axes[0, 0]
ax.plot(x_true, y_true, 'g--', linewidth=2, label='Ground truth')
ax.scatter(x_noisy, y_noisy, s=50, c='red', alpha=0.6, label='Observations')
ax.plot(x_noisy, y_noisy, 'r-', alpha=0.3)
ax.set_title('Without Temporal Constraint (闪烁)', fontsize=11, fontweight='bold')
ax.set_aspect('equal')
ax.legend()
ax.grid(True, alpha=0.3)

# 一阶平滑约束（拉普拉斯平滑）
ax = axes[0, 1]
x_smooth1 = np.zeros_like(x_noisy)
y_smooth1 = np.zeros_like(y_noisy)
x_smooth1[0] = x_noisy[0]
y_smooth1[0] = y_noisy[0]
for i in range(1, len(t)):
    x_smooth1[i] = 0.7 * x_noisy[i] + 0.3 * x_smooth1[i-1]
    y_smooth1[i] = 0.7 * y_noisy[i] + 0.3 * y_smooth1[i-1]

ax.plot(x_true, y_true, 'g--', linewidth=2, label='Ground truth')
ax.scatter(x_noisy, y_noisy, s=30, c='red', alpha=0.3, label='Observations')
ax.plot(x_smooth1, y_smooth1, 'b-', linewidth=2, label='With 1st-order smoothness')
ax.set_title('With Temporal Constraint (一阶平滑)', fontsize=11, fontweight='bold')
ax.set_aspect('equal')
ax.legend()
ax.grid(True, alpha=0.3)

# 损失函数对比
ax = axes[1, 0]
methods = ['No\nConstraint', '1st-order\nSmooth', '2nd-order\nSmooth']
photometric = [0.15, 0.12, 0.10]  # 光度误差
temporal = [0.8, 0.2, 0.05]       # 时序不一致性
total = np.array(photometric) + np.array(temporal)

x_pos = np.arange(len(methods))
width = 0.35

bars1 = ax.bar(x_pos - width/2, photometric, width, label='Photometric', color='steelblue', alpha=0.7)
bars2 = ax.bar(x_pos + width/2, temporal, width, label='Temporal inconsistency', color='coral', alpha=0.7)

ax.set_ylabel('Loss Value', fontweight='bold')
ax.set_title('Loss Decomposition', fontsize=11, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(methods)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 轨迹平滑度
ax = axes[1, 1]
# 计算一阶差分（速度）
vel_noisy = np.sqrt(np.diff(x_noisy)**2 + np.diff(y_noisy)**2)
vel_smooth1 = np.sqrt(np.diff(x_smooth1)**2 + np.diff(y_smooth1)**2)
vel_true = np.sqrt(np.diff(x_true)**2 + np.diff(y_true)**2)

time_steps = np.arange(1, len(t))
ax.plot(time_steps, vel_true, 'g--', linewidth=2, label='Ground truth velocity')
ax.plot(time_steps, vel_noisy, 'r-', alpha=0.5, linewidth=1.5, label='Noisy observations')
ax.plot(time_steps, vel_smooth1, 'b-', linewidth=2, label='With smoothness constraint')

ax.set_xlabel('Time Step', fontweight='bold')
ax.set_ylabel('Velocity Magnitude', fontweight='bold')
ax.set_title('Velocity Smoothness', fontsize=11, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n时序正则化的效果：")
print("✓ 无约束：高度匹配观察，但轨迹抖动（闪烁）")
print("✓ 一阶约束：平衡光度误差和平滑性")
print("✓ 二阶约束：更强的平滑性，但可能过度平滑")

## 4. 动态物体检测与分离

### 问题：静动混合场景

现实场景中通常同时有：
- **静态背景**：房间、建筑
- **动态前景**：人、车、移动物体

两者混合时，简单的平滑约束不够好——动态物体会被约束得太死板，而静态背景优化不精确。

### 解决方案 1: 光流引导

```
从连续帧中估计光流
     ↓
光流大的区域 → 动态
光流小的区域 → 静态
     ↓
分别优化不同约束
```

### 解决方案 2: 高斯分割

```
每个 Gaussian 预测一个动态系数 d ∈ [0, 1]

d = 0: 静态  → 强平滑约束
d = 1: 动态  → 无约束

学习 d 使得总损失最小
```

### 解决方案 3: 多物体跟踪

```
显式检测物体（如人、车）
     ↓
为每个物体维护独立的 Gaussians
     ↓
独立优化每个物体的运动
     ↓
背景 + 前景融合渲染
```

## 5. 4D Gaussian Splatting 详解

### 概念

4D GS 是时空对称的高斯表示，直接在 4D 空间中定义：

```
4D 点：p = (x, y, z, t) ∈ ℝ^4

4D 高斯：
  μ = (μ_x, μ_y, μ_z, μ_t) ∈ ℝ^4
  Σ ∈ ℝ^4×4  （4×4 协方差矩阵）

给定时刻 t，投影到 3D：
  取高斯的 (x,y,z) 分量
  约束：|t - μ_t| < 3σ_t
```

### 优势

1. **统一框架**：空间和时间对称处理
2. **连续表示**：自然的时间插值
3. **自动平滑**：协方差矩阵编码了时间平滑性

### 挑战

1. **参数爆炸**：4×4 矩阵而非 3×3
2. **光栅化复杂**：需要 4D→2D 投影
3. **数值稳定性**：4D 协方差矩阵可能病态

## 6. 总结

### 关键要点

1. **三种动态表示**：
   - 逐帧高斯：灵活但无时序约束
   - 变形场：参数少但难处理拓扑变化
   - 4D 高斯：统一框架但复杂

2. **时序约束至关重要**：
   - 平滑性、刚体、物理合理性
   - 权衡光度误差和时序一致

3. **动静分离是关键**：
   - 不同区域不同约束
   - 光流、分割或多物体跟踪

### 下一步

在 **[03_large_scale_scenes.ipynb](./03_large_scale_scenes.ipynb)** 中，我们将探讨如何处理大规模场景。

---